# Fusion_PSSM1110 训练 + 新发现 Acr 泛化验证

- **训练**：与 `protein_bert/fusion_confusion_matrix_demo.ipynb` 完全一致（同一数据、FusionTrainConfig、划分与流程），在 protein_bert 的 train/test 上训练，得到模型与 StandardScaler。
- **验证**：对新发现的 Acr（有序列且已跑完 PSSM 流水线）用训练时的 scaler 变换并预测，统计被正确判为 Acr 的比例等。

**注意**：将 `protein_bert` 加入 Python 路径；使用 conda 环境 `tf24pb`。

In [1]:
import os
import sys

# 本项目与 protein_bert 路径
PROJECT_ROOT = "/home/nemophila/projects/anticrispr_generalization"
PB_ROOT = "/home/nemophila/projects/protein_bert"
sys.path.insert(0, PB_ROOT)
os.chdir(PROJECT_ROOT)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, average_precision_score
from tensorflow import keras

from proteinbert import (
    load_anticrispr_with_ids,
    load_pretrained_model,
    FusionTrainConfig,
    load_feature_cache,
    attach_pssm_features,
    OutputSpec,
    OutputType,
    FinetuningModelGenerator,
    finetune,
)
from proteinbert.pssm_fusion import _build_late_fusion_model, _encode_x, find_best_threshold
from proteinbert.conv_and_global_attention_model import get_model_with_hidden_layers_as_outputs

BENCHMARKS_DIR = f"{PB_ROOT}/anticrispr_benchmarks"
PB_WORK = f"{PB_ROOT}/pssm_work"
SEED = 22
variant = "1110"

train_base_df, test_base_df = load_anticrispr_with_ids(BENCHMARKS_DIR, benchmark_name="anticrispr_binary")
parquet_path = f"{PB_WORK}/features/pssm_features_{variant}.parquet"
csv_path = f"{PB_WORK}/features/pssm_features_{variant}.csv"
cache_path = parquet_path if os.path.exists(parquet_path) else csv_path
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"PSSM cache not found: {cache_path}")
feature_df, feature_cols = load_feature_cache(cache_path)
train_df = attach_pssm_features(train_base_df, feature_df, feature_cols)
test_df = attach_pssm_features(test_base_df, feature_df, feature_cols)
y_test = test_df["label"].astype(int).to_numpy()

pmg, enc = load_pretrained_model(
    local_model_dump_dir=f"{PB_ROOT}/proteinbert_models",
    download_model_dump_if_not_exists=True,
    validate_downloading=False,
)
print("train_df:", train_df.shape, "test_df:", test_df.shape, "feature_cols:", len(feature_cols))

2026-02-23 14:07:01.207809: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


train_df: (1107, 1113) test_df: (286, 1113) feature_cols: 1110


## 训练 Fusion_PSSM1110，保存模型与 scaler

In [2]:
cfg = FusionTrainConfig(
    seq_len=512,
    batch_size=8,
    frozen_epochs=6,
    unfrozen_epochs=12,
    frozen_lr=1e-4,
    unfrozen_lr=2e-5,
    pssm_dropout=0.3,
    global_dropout=0.3,
    pssm_hidden_dim=128,
    global_hidden_dim=128,
    global_bottleneck_dim=64,
    fusion_hidden_dim=128,
    use_hidden_global_concat=True,
)

rng_train, rng_valid = train_test_split(
    train_df, test_size=0.1, stratify=train_df["label"], random_state=SEED
)
x_train = rng_train[feature_cols].to_numpy(dtype=np.float32)
x_valid = rng_valid[feature_cols].to_numpy(dtype=np.float32)
x_test = test_df[feature_cols].to_numpy(dtype=np.float32)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_valid = scaler.transform(x_valid)
x_test = scaler.transform(x_test)

y_train = rng_train["label"].astype(int).to_numpy()
y_valid = rng_valid["label"].astype(int).to_numpy()

X_train = _encode_x(enc, rng_train["seq"].tolist(), cfg.seq_len, x_train)
X_valid = _encode_x(enc, rng_valid["seq"].tolist(), cfg.seq_len, x_valid)
X_test = _encode_x(enc, test_df["seq"].tolist(), cfg.seq_len, x_test)

print("Training Fusion_PSSM1110 (seed=22)...")
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=cfg.patience, restore_best_weights=True
    )
]
model = _build_late_fusion_model(
    pmg, seq_len=cfg.seq_len, pssm_dim=len(feature_cols),
    freeze_pretrained_layers=True, cfg=cfg,
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cfg.frozen_lr),
    loss="binary_crossentropy",
)
model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=cfg.frozen_epochs,
    batch_size=cfg.batch_size,
    callbacks=callbacks,
    verbose=0,
)
for layer in model.layers:
    layer.trainable = True
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cfg.unfrozen_lr),
    loss="binary_crossentropy",
)
model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=cfg.unfrozen_epochs,
    batch_size=cfg.batch_size,
    callbacks=callbacks,
    verbose=0,
)

valid_prob = model.predict(X_valid, batch_size=cfg.batch_size, verbose=0).reshape(-1)
best_thr = find_best_threshold(y_valid, valid_prob)
y_prob_test = model.predict(X_test, batch_size=cfg.batch_size, verbose=0).reshape(-1)
y_pred_test = (y_prob_test >= best_thr).astype(int)
print(f"Best threshold (valid): {best_thr:.3f}")

os.makedirs(f"{PROJECT_ROOT}/outputs/saved_model", exist_ok=True)
model.save(f"{PROJECT_ROOT}/outputs/saved_model")
np.save(f"{PROJECT_ROOT}/outputs/scaler_fit.npy", {
    "mean_": scaler.mean_, "scale_": scaler.scale_, "best_thr": best_thr
})
print("Saved model and scaler to outputs/")

Training Fusion_PSSM1110 (seed=22)...


2026-02-23 14:07:02.783729: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-02-23 14:07:02.784740: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-02-23 14:07:02.805015: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:2a:00.0 name: NVIDIA L40S computeCapability: 8.9
coreClock: 2.52GHz coreCount: 142 deviceMemorySize: 44.53GiB deviceMemoryBandwidth: 804.75GiB/s
2026-02-23 14:07:02.805045: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-02-23 14:07:02.807041: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-02-23 14:07:02.807113: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2026-02-2

Best threshold (valid): 0.100


2026-02-23 14:08:57.725626: W tensorflow/python/util/util.cc:348] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.


INFO:tensorflow:Assets written to: /home/nemophila/projects/anticrispr_generalization/outputs/saved_model/assets
Saved model and scaler to outputs/


## 测试集混淆矩阵（与 demo 一致）

In [3]:
auc = roc_auc_score(y_test, y_prob_test)
acc = accuracy_score(y_test, y_pred_test)
auprc = average_precision_score(y_test, y_prob_test)
print("Fusion_PSSM1110 (seed=22) — Test set metrics")
print(f"  AUC:   {auc:.4f}")
print(f"  ACC:   {acc:.4f}")
print(f"  AUPRC: {auprc:.4f}")
cm = confusion_matrix(y_test, y_pred_test)
labels = ["Non-Acr", "Acr"]
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = "True"
cm_df.columns.name = "Predicted"
print("\nConfusion matrix")
display(cm_df)

Fusion_PSSM1110 (seed=22) — Test set metrics
  AUC:   0.9293
  ACC:   0.8427
  AUPRC: 0.7137

Confusion matrix


Predicted,Non-Acr,Acr
True,,
Non-Acr,219,41
Acr,4,22


## 新发现 Acr 泛化验证

加载本项目 `pssm_work/features` 下 1110 维 PSSM 缓存，用上面 fit 的 scaler 变换后预测。所有新 Acr 真实标签为 1，统计被预测为 Acr 的比例（即 Recall）。

In [4]:
NEW_WORK = f"{PROJECT_ROOT}/pssm_work"
NEW_CACHE = f"{NEW_WORK}/features/pssm_features_1110.parquet"
NEW_CACHE_CSV = f"{NEW_WORK}/features/pssm_features_1110.csv"
MANIFEST_PATH = f"{NEW_WORK}/sample_manifest.csv"

if not os.path.exists(MANIFEST_PATH):
    print("No manifest. Run: python scripts/prepare_fasta_manifest.py")
elif not os.path.exists(NEW_CACHE) and not os.path.exists(NEW_CACHE_CSV):
    print("No new Acr PSSM cache. Run: bash scripts/run_pssm_pipeline.sh")
else:
    new_cache_path = NEW_CACHE if os.path.exists(NEW_CACHE) else NEW_CACHE_CSV
    new_feat_df, _ = load_feature_cache(new_cache_path)
    manifest_df = pd.read_csv(MANIFEST_PATH)
    new_df = manifest_df.merge(new_feat_df, on="sample_id", how="inner")
    if new_df.empty:
        print("No overlapping sample_id between manifest and feature cache.")
    else:
        x_new = new_df[feature_cols].to_numpy(dtype=np.float32)
        x_new = scaler.transform(x_new)
        # Encoder only pads, does not truncate; truncate to seq_len-2 aa so token length <= seq_len
        max_aa = cfg.seq_len - 2
        seqs_new = [str(s)[:max_aa] if len(str(s)) > max_aa else str(s) for s in new_df["seq"].tolist()]
        X_new = _encode_x(enc, seqs_new, cfg.seq_len, x_new)
        prob_new = model.predict(X_new, batch_size=cfg.batch_size, verbose=0).reshape(-1)
        pred_new = (prob_new >= best_thr).astype(int)
        new_df = new_df.copy()
        new_df["score"] = prob_new
        new_df["predicted"] = pred_new
        display_cols = ["name", "sample_id", "score", "predicted"]
        if "name" not in new_df.columns:
            display_cols = [c for c in display_cols if c in new_df.columns]
        print("New Acr — prediction (true label = Acr for all)")
        display(new_df[display_cols])
        recall = pred_new.mean()
        n = len(pred_new)
        print(f"\nCorrectly predicted as Acr: {pred_new.sum()}/{n} — Recall = {recall:.4f}")

New Acr — prediction (true label = Acr for all)


,name,sample_id,score,predicted
0,AcrIA1,new_acr_AcrIA1,0.028796,0
1,AcrIE9,new_acr_AcrIE9,0.096483,0
2,AcrIIA24,new_acr_AcrIIA24,0.810308,1
3,AcrIIA25,new_acr_AcrIIA25,0.354682,1
4,AcrIIA26,new_acr_AcrIIA26,0.823331,1
5,AcrIIA27,new_acr_AcrIIA27,0.355460,1
6,AcrIIA28,new_acr_AcrIIA28,0.486737,1
7,AcrIIA29,new_acr_AcrIIA29,0.140269,1
8,AcrIIA30,new_acr_AcrIIA30,0.206595,1
9,AcrIIA31,new_acr_AcrIIA31,0.476824,1



Correctly predicted as Acr: 9/11 — Recall = 0.8182


## 仅用 1110 维 PSSM 特征

与 demo 中 Ablation_RPSSM_1110 逻辑一致：仅用 1110 维 PSSM 训练 LogisticRegression，在验证集上找最优阈值，输出测试集 AUC/ACC/AUPRC、混淆矩阵，以及对 11 个新蛋白的预测。

In [5]:
# 与 Fusion 使用相同划分 (rng_train, rng_valid 已在上面定义)
x_tr = rng_train[feature_cols].to_numpy(dtype=np.float32)
x_va = rng_valid[feature_cols].to_numpy(dtype=np.float32)
x_te = test_df[feature_cols].to_numpy(dtype=np.float32)
scaler_pssm = StandardScaler()
x_tr = scaler_pssm.fit_transform(x_tr)
x_va = scaler_pssm.transform(x_va)
x_te = scaler_pssm.transform(x_te)
y_va = rng_valid["label"].astype(int).to_numpy()

clf_pssm = LogisticRegression(max_iter=2000, solver="liblinear", random_state=SEED)
clf_pssm.fit(x_tr, rng_train["label"].astype(int).to_numpy())
va_prob_pssm = clf_pssm.predict_proba(x_va)[:, 1]
best_thr_pssm = find_best_threshold(y_va, va_prob_pssm)
y_prob_pssm = clf_pssm.predict_proba(x_te)[:, 1]
y_pred_pssm = (y_prob_pssm >= best_thr_pssm).astype(int)

auc_pssm = roc_auc_score(y_test, y_prob_pssm)
acc_pssm = accuracy_score(y_test, y_pred_pssm)
auprc_pssm = average_precision_score(y_test, y_prob_pssm)
print("PSSM-only (1110 dim, seed=22) — Test set metrics")
print(f"  Best threshold (valid): {best_thr_pssm:.3f}")
print(f"  AUC:   {auc_pssm:.4f}")
print(f"  ACC:   {acc_pssm:.4f}")
print(f"  AUPRC: {auprc_pssm:.4f}")
cm_pssm = confusion_matrix(y_test, y_pred_pssm)
cm_pssm_df = pd.DataFrame(cm_pssm, index=["Non-Acr", "Acr"], columns=["Non-Acr", "Acr"])
cm_pssm_df.index.name = "True"
cm_pssm_df.columns.name = "Predicted"
print("\nConfusion matrix")
display(cm_pssm_df)

PSSM-only (1110 dim, seed=22) — Test set metrics
  AUC:   0.8649
  ACC:   0.8706
  AUPRC: 0.4662

Confusion matrix


Predicted,Non-Acr,Acr
True,,
Non-Acr,234,26
Acr,11,15


In [6]:
NEW_WORK = f"{PROJECT_ROOT}/pssm_work"
NEW_CACHE = f"{NEW_WORK}/features/pssm_features_1110.parquet"
NEW_CACHE_CSV = f"{NEW_WORK}/features/pssm_features_1110.csv"
MANIFEST_PATH = f"{NEW_WORK}/sample_manifest.csv"

# PSSM-only：11 个新蛋白预测
if not os.path.exists(MANIFEST_PATH):
    print("No manifest. Run: python scripts/prepare_fasta_manifest.py")
elif not os.path.exists(NEW_CACHE) and not os.path.exists(NEW_CACHE_CSV):
    print("No new Acr PSSM cache. Run: bash scripts/run_pssm_pipeline.sh")
else:
    new_cache_path = NEW_CACHE if os.path.exists(NEW_CACHE) else NEW_CACHE_CSV
    new_feat_df, _ = load_feature_cache(new_cache_path)
    manifest_df = pd.read_csv(MANIFEST_PATH)
    new_df_pssm = manifest_df.merge(new_feat_df, on="sample_id", how="inner")
    if new_df_pssm.empty:
        print("No overlapping sample_id between manifest and feature cache.")
    else:
        x_new_pssm = new_df_pssm[feature_cols].to_numpy(dtype=np.float32)
        x_new_pssm = scaler_pssm.transform(x_new_pssm)
        prob_new_pssm = clf_pssm.predict_proba(x_new_pssm)[:, 1]
        pred_new_pssm = (prob_new_pssm >= best_thr_pssm).astype(int)
        new_df_pssm = new_df_pssm.copy()
        new_df_pssm["score"] = prob_new_pssm
        new_df_pssm["predicted"] = pred_new_pssm
        display_cols = ["name", "sample_id", "score", "predicted"]
        if "name" not in new_df_pssm.columns:
            display_cols = [c for c in display_cols if c in new_df_pssm.columns]
        print("PSSM-only — New Acr prediction (true label = Acr for all)")
        display(new_df_pssm[display_cols])
        recall_pssm = pred_new_pssm.mean()
        n = len(pred_new_pssm)
        print(f"\nCorrectly predicted as Acr: {pred_new_pssm.sum()}/{n} — Recall = {recall_pssm:.4f}")

PSSM-only — New Acr prediction (true label = Acr for all)


,name,sample_id,score,predicted
0,AcrIA1,new_acr_AcrIA1,0.084606,0
1,AcrIE9,new_acr_AcrIE9,0.921290,1
2,AcrIIA24,new_acr_AcrIIA24,0.931310,1
3,AcrIIA25,new_acr_AcrIIA25,0.221753,0
4,AcrIIA26,new_acr_AcrIIA26,0.449106,0
5,AcrIIA27,new_acr_AcrIIA27,0.027607,0
6,AcrIIA28,new_acr_AcrIIA28,0.805527,1
7,AcrIIA29,new_acr_AcrIIA29,0.001541,0
8,AcrIIA30,new_acr_AcrIIA30,0.167190,0
9,AcrIIA31,new_acr_AcrIIA31,0.506662,0



Correctly predicted as Acr: 3/11 — Recall = 0.2727


## 仅用 finetune 过的 ProteinBERT

与 demo 中 Stage1 逻辑一致：仅用序列 finetune ProteinBERT（无 PSSM），在验证集上找最优阈值，输出测试集 AUC/ACC/AUPRC、混淆矩阵，以及对 11 个新蛋白的预测。

In [7]:
output_type = OutputType(False, "binary")
output_spec = OutputSpec(output_type, [0, 1])
mg_bert = FinetuningModelGenerator(
    pmg,
    output_spec=output_spec,
    pretraining_model_manipulation_function=get_model_with_hidden_layers_as_outputs,
    dropout_rate=0.3,
    head_type="classification",
    loss_type="bce",
    lr=1e-4,
)
finetune(
    mg_bert,
    enc,
    output_spec,
    rng_train["seq"].tolist(),
    rng_train["label"].tolist(),
    rng_valid["seq"].tolist(),
    rng_valid["label"].tolist(),
    seq_len=512,
    batch_size=8,
    max_epochs_per_stage=8,
    begin_with_frozen_pretrained_layers=True,
    n_final_epochs=0,
)
model_bert = mg_bert.create_model(512)
X_valid_bert = enc.encode_X(rng_valid["seq"].tolist(), 512)
valid_prob_bert = model_bert.predict(X_valid_bert, batch_size=8, verbose=0).reshape(-1)
best_thr_bert = find_best_threshold(y_valid, valid_prob_bert)
X_test_bert = enc.encode_X(test_df["seq"].tolist(), 512)
y_prob_bert = model_bert.predict(X_test_bert, batch_size=8, verbose=0).reshape(-1)
y_pred_bert = (y_prob_bert >= best_thr_bert).astype(int)

auc_bert = roc_auc_score(y_test, y_prob_bert)
acc_bert = accuracy_score(y_test, y_pred_bert)
auprc_bert = average_precision_score(y_test, y_prob_bert)
print("ProteinBERT-only (finetune, seed=22) — Test set metrics")
print(f"  Best threshold (valid): {best_thr_bert:.3f}")
print(f"  AUC:   {auc_bert:.4f}")
print(f"  ACC:   {acc_bert:.4f}")
print(f"  AUPRC: {auprc_bert:.4f}")
cm_bert = confusion_matrix(y_test, y_pred_bert)
cm_bert_df = pd.DataFrame(cm_bert, index=["Non-Acr", "Acr"], columns=["Non-Acr", "Acr"])
cm_bert_df.index.name = "True"
cm_bert_df.columns.name = "Predicted"
print("\nConfusion matrix")
display(cm_bert_df)

[2026_02_23-14:09:12] Training set: Filtered out 0 of 996 (0.0%) records of lengths exceeding 510.
[2026_02_23-14:09:12] Validation set: Filtered out 0 of 111 (0.0%) records of lengths exceeding 510.
[2026_02_23-14:09:12] Training with frozen pretrained layers...
Epoch 1/8
125/125 [==============================] - 9s 34ms/step - loss: 0.5930 - val_loss: 0.4896
Epoch 2/8
125/125 [==============================] - 3s 24ms/step - loss: 0.4691 - val_loss: 0.4642
Epoch 3/8
125/125 [==============================] - 3s 24ms/step - loss: 0.4195 - val_loss: 0.4453
Epoch 4/8
125/125 [==============================] - 3s 24ms/step - loss: 0.3969 - val_loss: 0.4310
Epoch 5/8
125/125 [==============================] - 3s 25ms/step - loss: 0.3933 - val_loss: 0.4176
Epoch 6/8
125/125 [==============================] - 3s 24ms/step - loss: 0.3924 - val_loss: 0.4063
Epoch 7/8
125/125 [==============================] - 3s 24ms/step - loss: 0.4031 - val_loss: 0.3976
Epoch 8/8
125/125 [=================

Predicted,Non-Acr,Acr
True,,
Non-Acr,212,48
Acr,6,20


In [8]:
NEW_WORK = f"{PROJECT_ROOT}/pssm_work"
NEW_CACHE = f"{NEW_WORK}/features/pssm_features_1110.parquet"
NEW_CACHE_CSV = f"{NEW_WORK}/features/pssm_features_1110.csv"
MANIFEST_PATH = f"{NEW_WORK}/sample_manifest.csv"

# ProteinBERT-only：11 个新蛋白预测（仅需序列，与上面 NEW_WORK / MANIFEST 一致）
if not os.path.exists(MANIFEST_PATH):
    print("No manifest. Run: python scripts/prepare_fasta_manifest.py")
elif not os.path.exists(NEW_CACHE) and not os.path.exists(NEW_CACHE_CSV):
    print("No new Acr PSSM cache. Run: bash scripts/run_pssm_pipeline.sh")
else:
    new_cache_path = NEW_CACHE if os.path.exists(NEW_CACHE) else NEW_CACHE_CSV
    new_feat_df, _ = load_feature_cache(new_cache_path)
    manifest_df = pd.read_csv(MANIFEST_PATH)
    new_df_bert = manifest_df.merge(new_feat_df, on="sample_id", how="inner")
    if new_df_bert.empty:
        print("No overlapping sample_id between manifest and feature cache.")
    else:
        max_aa = 512 - 2
        seqs_new_bert = [
            str(s)[:max_aa] if len(str(s)) > max_aa else str(s)
            for s in new_df_bert["seq"].tolist()
        ]
        X_new_bert = enc.encode_X(seqs_new_bert, 512)
        prob_new_bert = model_bert.predict(X_new_bert, batch_size=8, verbose=0).reshape(-1)
        pred_new_bert = (prob_new_bert >= best_thr_bert).astype(int)
        new_df_bert = new_df_bert.copy()
        new_df_bert["score"] = prob_new_bert
        new_df_bert["predicted"] = pred_new_bert
        display_cols = ["name", "sample_id", "score", "predicted"]
        if "name" not in new_df_bert.columns:
            display_cols = [c for c in display_cols if c in new_df_bert.columns]
        print("ProteinBERT-only — New Acr prediction (true label = Acr for all)")
        display(new_df_bert[display_cols])
        recall_bert = pred_new_bert.mean()
        n = len(pred_new_bert)
        print(f"\nCorrectly predicted as Acr: {pred_new_bert.sum()}/{n} — Recall = {recall_bert:.4f}")

ProteinBERT-only — New Acr prediction (true label = Acr for all)


,name,sample_id,score,predicted
0,AcrIA1,new_acr_AcrIA1,0.000470,0
1,AcrIE9,new_acr_AcrIE9,0.044963,0
2,AcrIIA24,new_acr_AcrIIA24,0.913880,1
3,AcrIIA25,new_acr_AcrIIA25,0.238588,1
4,AcrIIA26,new_acr_AcrIIA26,0.996046,1
5,AcrIIA27,new_acr_AcrIIA27,0.304343,1
6,AcrIIA28,new_acr_AcrIIA28,0.926779,1
7,AcrIIA29,new_acr_AcrIIA29,0.012732,0
8,AcrIIA30,new_acr_AcrIIA30,0.082603,1
9,AcrIIA31,new_acr_AcrIIA31,0.882939,1



Correctly predicted as Acr: 7/11 — Recall = 0.6364
